# Monte Carlo Trajectory Failure Rate

This notebook samples many closed-loop trajectories using `run_agent_env_sensor_loop` and estimates the empirical failure rate.

Failure is defined here as `terminated_early == True` (either specification violation or simulator constraint violation).

In [8]:
import io
import contextlib
from collections import Counter
from tqdm import tqdm

import numpy as np

from main import run_agent_env_sensor_loop

In [ ]:
# --- Simulation configuration ---
N_ROLLOUTS = 100
BASE_SEED = 0

Z0 = 0.8
Z_TARGET = 0.8
N_STEPS = 2000
ENABLE_LATERAL_DAMPER = True
USE_EKF_PID_CONTROLLER = True
GROUND_EFFECT_ENABLED = True

SENSOR_ARGS = dict(
    mu_A=0.1,
    sigma_A=0.05,
    mu_k=2 * np.pi,
    sigma_k=0.5,
    sigma_eps=0.01,
    p_penetration=0.05,
    alpha_min=0.10,
    alpha_max=10,
    perfect_sensing=False,
)

SPECIFICATION = dict(
    z_min=0.2,
    z_max=1.2,
    pitch_min=-np.deg2rad(15.0),
    pitch_max=np.deg2rad(15.0),
)

In [10]:
def sample_trajectories(n_rollouts, base_seed=0):
    records = []

    for i in tqdm(range(n_rollouts)):
        seed = base_seed + i

        # Suppress per-step prints from the rollout function for clean Monte Carlo runs
        with contextlib.redirect_stdout(io.StringIO()):
            result = run_agent_env_sensor_loop(
                z0=Z0,
                z_target=Z_TARGET,
                n_steps=N_STEPS,
                seed=seed,
                enable_lateral_damper=ENABLE_LATERAL_DAMPER,
                use_ekf_pid_controller=USE_EKF_PID_CONTROLLER,
                sensor_args=SENSOR_ARGS,
                specification=SPECIFICATION,
                save_trajectory_plot=False,
                ground_effect_enabled=GROUND_EFFECT_ENABLED,
            )

        failed = bool(result["terminated_early"])
        records.append(
            {
                "seed": seed,
                "failed": failed,
                "termination_type": result["termination_type"],
                "termination_step": result["termination_step"],
            }
        )

    return records

In [11]:
records = sample_trajectories(N_ROLLOUTS, BASE_SEED)

n_total = len(records)
n_fail = sum(r["failed"] for r in records)
failure_rate = n_fail / n_total if n_total > 0 else float("nan")

termination_counts = Counter(
    (r["termination_type"] if r["failed"] else "no_failure")
    for r in records
)

print(f"Total rollouts:  {n_total}")
print(f"Failed rollouts: {n_fail}")
print(f"Failure rate:    {failure_rate:.4%}")

print("\nBreakdown by termination type:")
for t, c in termination_counts.most_common():
    print(f"  {t:>20}: {c}")

failure_steps = [r["termination_step"] for r in records if r["failed"] and r["termination_step"] is not None]
if failure_steps:
    print("\nFailure step stats:")
    print(f"  Mean: {np.mean(failure_steps):.2f}")
    print(f"  Min:  {np.min(failure_steps)}")
    print(f"  Max:  {np.max(failure_steps)}")

100%|██████████| 100/100 [10:56<00:00,  6.57s/it]

Total rollouts:  100
Failed rollouts: 0
Failure rate:    0.0000%

Breakdown by termination type:
            no_failure: 100


In [ ]:
def wilson_interval(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan")
    phat = k / n
    denom = 1 + z**2 / n
    center = (phat + z**2 / (2 * n)) / denom
    radius = z * np.sqrt((phat * (1 - phat) + z**2 / (4 * n)) / n) / denom
    return center - radius, center + radius

lo, hi = wilson_interval(n_fail, n_total)
print(f"95% Wilson CI for failure rate: [{lo:.4%}, {hi:.4%}]")